# Q-Learning Tabular - CartPole
Este notebook muestra cómo aplicar Q-learning clásico (tabular) en el entorno CartPole, discretizando los estados continuos.

In [ ]:
import numpy as np
import gymnasium as gym

# Discretización
bins = [
    np.linspace(-4.8, 4.8, 10),
    np.linspace(-5, 5, 10),
    np.linspace(-0.418, 0.418, 10),
    np.linspace(-5, 5, 10)
]

def discretize(obs):
    return tuple(
        np.digitize(val, binas) for val, binas in zip(obs, bins)
    )

In [ ]:
env = gym.make("CartPole-v1")
q_table = np.zeros((10, 10, 10, 10, env.action_space.n))

In [ ]:
# Parámetros
alpha = 0.1
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01

In [ ]:
# Entrenamiento
for episode in range(1000):
    obs, _ = env.reset()
    state = discretize(obs)
    done = False
    total_reward = 0
    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        new_obs, reward, terminated, truncated, _ = env.step(action)
        new_state = discretize(new_obs)
        done = terminated or truncated

        best_future = np.max(q_table[new_state])
        q_table[state][action] += alpha * (reward + gamma * best_future - q_table[state][action])

        state = new_state
        total_reward += reward

    epsilon = max(epsilon * epsilon_decay, epsilon_min)
    if episode % 100 == 0:
        print(f"Episodio {episode}, recompensa: {total_reward}")

In [ ]:
# Evaluación
for episode in range(5):
    obs, _ = env.reset()
    state = discretize(obs)
    done = False
    while not done:
        env.render()
        action = np.argmax(q_table[state])
        obs, _, terminated, truncated, _ = env.step(action)
        state = discretize(obs)
        done = terminated or truncated
env.close()